# Creating a 3D model of an FD-SOI quantum dot with QTCAD® Builder

**An extensive tutorial for designing geometries Using QTCAD® Builder**

## 1. Introduction

In this tutorial, we study a single fully depleted silicon-on-insulator (FD-SOI) device, a widely used architecture in nanoscale electronics.

A recent industrial demonstration of this technology is presented in [‘Commercial CMOS Process for Quantum Computing: Quantum Dots and Charge Sensing in a 22 nm Fully Depleted Silicon-on-Insulator Process’ 🔗](https://arxiv.org/abs/2412.08422v1). IEEE Electron Device Lett. 46, 1913–1916 (2025). doi:10.1109/LED.2025.3595384.

The device consists of:
- A silicon channel where a quantum dot is confined.
- Two doped regions that act as the **source** and **drain**;
- A gate electrode (called the **plunger gate**) used to confine electrons in the channel;
- Two additional **barrier gates** that provide confinement along the channel direction;
- An oxide layer surrounding above and below the silicon channel, providing confinement in the perpendicular direction.

The charge carriers in this device are **electrons**. These electrons are confined:
- **Laterally (along the channel)** by the barrier gates;  
- **Vertically (perpendicular to the channel)** by the surrounding oxide.

<div>
<img src="figures/sqfdsoi.png" width="500px"/>
</div>

#### Material Stack

The heterostructure of the device consists of:

- 2 nm of HfO₂ on top (high-k dielectric for gate insulation)
- 10 nm silicon layer
- 10 nm buried oxide

This layered structure enables precise control over the quantum dot’s confinement both laterally and vertically, making it suitable for nanoscale and quantum device simulations.

Furthermore, in QTCAD® there is a concept called a **dot region**. In the dot region, charges are treated quantum mechanically and are solved using our quantum mechanical solvers, such as the Schrödinger solver.

Since we expect the quantum dot to form under the plunger gate, we define the dot region as a volume in the silicon channel that includes all regions expected to be relevant to the quantum dot. This region is shown as a dotted box in the following figure.

<div>
<img src="figures/dotr.png" width="500px"/>
</div>

## 2. Device geometry

The device geometry can be created by designing the layout in KLayout and exporting it in the OAS format, which can then be imported into QTCAD®. QTCAD® Builder natively supports these layout files, streamlining the workflow from layout design to simulation.

<div class="alert alert-block alert-info">

**Note**

KLayout is not included with QTCAD®. If you wish to create your own layout, you must install KLayout separately. Detailed installation instructions and tutorials for creating device layouts can be found on the [KLayout official website](https://www.klayout.de/build.html).

For the purposes of this notebook, we provide a predefined `.oas` file, so students can skip the layout creation entirely. Sections 2.1–2.5 demonstrate how to build the geometry from scratch and are **optional**. They are intended for students who want to understand the process in detail or create custom device layouts.
</div>


<div class="alert alert-block alert-info" style="background: #F4F4F4">

**Optional exercise:** 

If you want to explore creating the device yourself:

- Install KLayout following the instructions linked above.
- Open the layout editor and create the device geometry.
- Export the layout as an `.oas` file.
- Import the `.oas` file into QTCAD® Builder.
</div>

### 2.1 Steps in KLayout (Optional)

### 2.1.1 Create a new layer

Start by creating a new layer in KLayout.


<div>
<img src="figures/layers.png" width="200px"/>
</div>

### 2.1.2 Draw the required geometries

Create all required shapes corresponding to the device components (channel, gates, source, drain, etc.).

<div>
<img src="figures/layout.png" width="500px"/>
</div>

### 2.1.3 Add a shape name properties

First, click on a shape to open its properties window.

<div>
<img src="figures/prop.png" width="500px"/>
</div>

Then, in the properties window, select the option to add properties.

As shown in the figure, navigate to the list. In this list, each item should be a key–value pair where the key is "name" and the value is the name of the shape they want to add. Repeat this step for each shape.
<div>
<img src="figures/name.png" width="500px"/>
</div>

### 2.1.4 Purpose of the shape name

The assigned property serves as an identifier that will later be used in QTCAD® for:

- Boundary tags  
- Volume tags  
- Applying materials  
- Applying boundary conditions  

---

After assigning properties to all shapes, save the layout in the `.oas` format.

## 3. Design

We now create the final 3D design in QTCAD® Builder using the blueprint provided by the OAS file.

First, we import the relevant libraries and define the paths for the `.oas` file as well as the destination `.msh` (mesh) and `.xao` (geometry) files. The `.xao` geometry file is needed because it stores the precise 3D shape of the device, which QTCAD® uses for adaptive meshing and accurate simulation of physical regions.

In [1]:
from qtcad.builder import Builder
from pathlib import Path

# Specify the paths to the layout, mesh and XAO files.
script_path = Path("__file__").parent.resolve()
layout_path = script_path / "layouts" / "sqdfdsoi.oas"
# Mesh file for the device.
mesh_path = script_path / "meshes" / "sqdfdsoi.msh"
# Geometry file for the device. It is used for adaptive meshing.
xao_path = script_path / "meshes" / "sqdfdsoi.xao"

Next, we define the dimensions of growth along the $z$-axis.

The FD-SOI structure consists of a $2$ nm oxide layer on top of the channel, a $10$ nm thick channel thickness (also referred to as the film), and an additional $10$ nm region extending below the channel.

In [2]:
gate_oxide_thick = 2
film_thick = 10
box_thick = 10

We also determine the characteristic length of the mesh entire device.

In [3]:
char_len = 10

### 3.1 Initialize a `Builder` object and load the layout

Next, we instantiate a `Builder` object and load the `.oas` file using the [`load_layout` method 🔗](https://docs.nanoacademic.com/qtcad/API_reference/qtcad.builder/#qtcad.builder.Builder.load_layout) of the `Builder` class.

In [4]:
bd = Builder(name="sqdfdsoi")
bd.load_layout(file_path=str(layout_path))


|  ||  |||                                                                 \\  
|  ||  |||         @@@@     @@@@@@@@      @@@@@        @       @@@@@@@      \\ 
|  ||  |||      @@@   @@@      @@      @@@    @       @@       @@    @@@     \\
|  ||  |||     @@       @@     @@     @@             @ @@      @@     @@@     \\
|  ||  |||     @@       @@     @@     @@            @    @     @@      @@     //
|  ||  |||      @@     @@      @@      @@@    @    @@@@@@@@    @@    @@@     //
|  ||  |||        @@@@@@       @@        @@@@@@   @@      @@   @@@@@@       // 
|  ||  |||             @@                                                  //   

                                 Version 2.1.3                                  
  Copyright (c) 2022-2026 Nanoacademic Technologies Inc. All rights reserved.   

      Welcome to QTCAD, the Quantum-Technology Computer-Aided Design tool.      

                        For documentation, please visit:                        
                      https:/

[10:16:56] INFO     Adding layer Domain                                                                ]8;id=593736;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\masks.py\masks.py]8;;\:]8;id=909019;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\masks.py#741\741]8;;\

           INFO     Adding layer DotRegion                                                             ]8;id=867623;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\masks.py\masks.py]8;;\:]8;id=180774;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\masks.py#741\741]8;;\

           INFO     Adding layer Film                                                                  ]8;id=447853;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\masks.py\masks.py]8;;\:]8;id=831668;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\masks.py#741\741]8;;\

           INFO     Adding layer TopGates                                                              ]8;id=154946;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\masks.py\masks.py]8;;\:]8;id=264475;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\masks.py#741\741]8;;\

### 3.2 Print the layout structure

To verify the structure of the loaded layout file, or to confirm the contents when a mask is in use, we can print the mask tree using the [`print_mask_tree` method 🔗](https://docs.nanoacademic.com/qtcad/API_reference/qtcad.builder/#qtcad.builder.Builder.print_mask_tree).
It displays all layers (as defined in KLayout) or masks (in Builder terminology).

Note that each mask consists of a set of polygons. Following the procedure described earlier, each polygon should have an assigned name. These names will be displayed alongside their corresponding polygons in the printed structure.

<div class="alert alert-block alert-info">

**Note**

In real device layouts, any unnamed or default polygons in the OAS file may cause ambiguities when imported into QTCAD®. If you encounter such polygons, you should return to the layout editor (e.g., KLayout) and assign appropriate names before importing the file.

</div>

In [5]:
bd.print_mask_tree()

Layout
├── Mask 1 "Domain"
│   └── 0  Polygon  "domain"
├── Mask 4 "DotRegion"
│   └── 0  Polygon  "dot_region"
├── Mask 2 "Film"
│   ├── 0  Polygon  "spacer_1"
│   ├── 1  Polygon  "source"
│   ├── 2  Polygon  "spacer_2"
│   ├── 3  Polygon  "drain"
│   └── 4  Polygon  "channel"
└── Mask 3 "TopGates"
    ├── 0  Polygon  "barrier_gate_2"
    ├── 1  Polygon  "plunger_gate"
    └── 2  Polygon  "barrier_gate_1"

### 3.3 Grow the structure from bottom to top

Let us then construct the 3D device structure by sequentially building layers from the bottom (BOX/Buried oxide layer) to the top (gate surfaces), using masks to define each layer.

To that end, we need to
1. **Define the device domain**
2. **Grow the buried oxide layer ("box")**
3. **Define the silicon film layer**
4. **Grow the gate oxide layer**
5. **Add the top gate surfaces**
6. **Add the back gate**

#### 3.3.1 Define the device domain

We start by working on the `"Domain"` mask, which defines the overall extent of the device, and by setting the characteristic length of the mesh.
We do this using the [`set_mesh_size` method 🔗](https://docs.nanoacademic.com/qtcad/API_reference/qtcad.builder/#qtcad.builder.Builder.set_mesh_size). It defines the spatial resolution for the simulation or meshing within the currently active domain.

In [6]:
bd.use_mask("Domain").set_mesh_size(char_len)

[10:17:13] INFO     Using mask 'Domain'                                                              ]8;id=40594;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=978737;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#373\373]8;;\

#### 3.3.2 Grow the buried oxide layer ("BOX")
To create the buried oxide layer with a thickness of `box_thick`, we copy the `"Domain"` mask, rename it as `"box"` and extrude it upwards.

In [7]:
bd.copy_mask("box").rename_shape("box").extrude(box_thick)

[10:17:18] INFO     Extruding shape 1 from mask 'box' with name 'box' by 10 at z=0                  ]8;id=797063;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=396501;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1359\1359]8;;\

[10:17:20] INFO     Creating new physical group with name 'box_bottom'                              ]8;id=323554;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=768515;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Creating new physical group with name 'box_side'                                ]8;id=900717;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=998822;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Creating new physical group with name 'box_top'                                 ]8;id=310556;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=441693;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Creating new physical group with name 'box'                                     ]8;id=842603;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=864745;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Setting z-coordinate to 10                                                       ]8;id=267619;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=693653;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#418\418]8;;\

#### 3.3.3 Define the silicon film layer

The `"Film"` mask represents the silicon channel. To create the 3D silicon layer above the "box", we extrude it by film_thick. This silicon film includes the channel as well as the source and drain regions.

Between the source, channel, and drain, spacers are added to physically separate these regions. The channel is the key region where the quantum dot is expected to be confined.

In [8]:
bd.use_mask("Film")
bd.extrude(film_thick)

[10:17:25] INFO     Using mask 'Film'                                                                ]8;id=575723;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=19841;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#373\373]8;;\

           INFO     Extruding shape 0 from mask 'Film' with name 'spacer_1' by 10 at z=10           ]8;id=190062;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=512189;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1359\1359]8;;\

           INFO     Creating new physical group with name 'spacer_1_bottom'                         ]8;id=273190;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=678675;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Creating new physical group with name 'spacer_1_side'                           ]8;id=98042;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=683870;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Creating new physical group with name 'spacer_1_top'                            ]8;id=110806;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=848229;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

...

           INFO     Creating new physical group with name 'channel_side'                            ]8;id=407463;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=144849;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Creating new physical group with name 'channel_top'                             ]8;id=277865;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=298425;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Fragmenting...                                                                ]8;id=453898;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py\fragmenter.py]8;;\:]8;id=928803;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py#105\105]8;;\

[10:17:38] INFO     Creating new physical group with name 'channel'                                 ]8;id=734690;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=975287;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Setting z-coordinate to 20                                                       ]8;id=359958;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=685713;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#418\418]8;;\

#### 3.3.4 Grow the gate oxide layer

Let us create the gate oxide layer using a `"gate_oxide"` mask copied from the domain. We then extrude it to the desired thickness, `gate_oxide_thick`.

In [9]:
bd.use_mask("Domain")
bd.copy_mask("gate_oxide").rename_shape("gate_oxide")
bd.extrude(gate_oxide_thick)

[10:17:48] INFO     Using mask 'Domain'                                                              ]8;id=234856;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=452572;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#373\373]8;;\

           INFO     Extruding shape 1 from mask 'gate_oxide' with name 'gate_oxide' by 2 at z=20    ]8;id=13063;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=94184;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1359\1359]8;;\

           INFO     Creating new physical group with name 'gate_oxide_bottom'                       ]8;id=302857;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=560905;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Creating new physical group with name 'gate_oxide_side'                         ]8;id=607419;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=741906;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Creating new physical group with name 'gate_oxide_top'                          ]8;id=842713;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=985194;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Fragmenting...                                                                ]8;id=461184;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py\fragmenter.py]8;;\:]8;id=68101;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py#105\105]8;;\

[10:17:51] INFO     Creating new physical group with name 'gate_oxide'                              ]8;id=186115;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=579810;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Setting z-coordinate to 22                                                       ]8;id=76338;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=428818;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#418\418]8;;\

#### 3.3.5 Add the top gate surfaces
The top gates are created from the `"TopGates"` mask and added on top of the gate oxide. Since they will function purely as a boundary condition, they are incorporated as surfaces rather than volumetric layers.

The [`add_surface` method 🔗](https://docs.nanoacademic.com/qtcad/API_reference/qtcad.builder/#qtcad.builder.Builder.add_surface) is used to add this mask as a surface that spans the entire domain at the base of the device.

In [10]:
bd.use_mask("TopGates")
bd.add_surface()

[10:17:55] INFO     Using mask 'TopGates'                                                            ]8;id=98195;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=233465;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#373\373]8;;\

           INFO     Creating a surface from shape 0 with name 'barrier_gate_2' from mask 'TopGates' ]8;id=701061;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=951833;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1212\1212]8;;\
                    at z=22                                                                                        

           INFO     Fragmenting...                                                                ]8;id=84971;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py\fragmenter.py]8;;\:]8;id=743782;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py#105\105]8;;\

           INFO     Creating new physical group with name 'barrier_gate_2'                          ]8;id=367024;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=773864;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

[10:17:56] INFO     Creating a surface from shape 1 with name 'plunger_gate' from mask 'TopGates'   ]8;id=646710;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=532599;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1212\1212]8;;\
                    at z=22                                                                                        

           INFO     Fragmenting...                                                                ]8;id=373278;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py\fragmenter.py]8;;\:]8;id=272381;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py#105\105]8;;\

           INFO     Creating new physical group with name 'plunger_gate'                            ]8;id=874385;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=95842;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

           INFO     Creating a surface from shape 2 with name 'barrier_gate_1' from mask 'TopGates' ]8;id=987245;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=758827;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1212\1212]8;;\
                    at z=22                                                                                        

           INFO     Fragmenting...                                                                ]8;id=311962;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py\fragmenter.py]8;;\:]8;id=608029;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py#105\105]8;;\

[10:17:57] INFO     Creating new physical group with name 'barrier_gate_1'                          ]8;id=47098;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=95176;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

#### 3.3.6 Add the back gate
Lastly, let us add the back gate. It is positioned at the bottom of the device ($z = 0$) using a `"back_gate"` mask copied from the domain.

Like the top gates, it is added to the geometry purely as a surface using the `add_surface` method, rather than as a volumetric layer.

In [11]:
bd.use_mask("Domain")
bd.set_z(0)
bd.copy_mask("back_gate").rename_shape("back_gate")
bd.add_surface()

[10:18:02] INFO     Using mask 'Domain'                                                              ]8;id=841732;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=882495;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#373\373]8;;\

           INFO     Setting z-coordinate to 0                                                        ]8;id=782802;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=988426;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#418\418]8;;\

           INFO     Creating a surface from shape 1 with name 'back_gate' from mask 'back_gate' at  ]8;id=896546;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=19258;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1212\1212]8;;\
                    z=0                                                                                            

           INFO     Fragmenting...                                                                ]8;id=683993;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py\fragmenter.py]8;;\:]8;id=18332;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\fragmenter.py#105\105]8;;\

[10:18:03] INFO     Creating new physical group with name 'back_gate'                               ]8;id=8779;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=716621;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1254\1254]8;;\

### 3.4 Generating and saving the mesh
Once the 3D device structure is defined, the next step is to generate a computational mesh.

In QTCAD® Builder, this is straightforward: you simply call the mesh method on the `Builder` object, specifying any relevant parameters. This creates a mesh that follows the geometry of your device, ready for QTCAD® simulations.

After the mesh is generated, you can save both the mesh and the geometry files using the [`write` method 🔗](https://docs.nanoacademic.com/qtcad/API_reference/qtcad.builder/#qtcad.builder.Builder.write). Typically, we save:
- A `.msh` file, which contains the mesh information;
- A `.xao` file, which stores the full 3D geometry of the device.

In [12]:
bd.mesh(3)
bd.write(mesh_path)
bd.write(xao_path)
bd.view()

[10:18:07] INFO     Preparing to mesh                                                                ]8;id=645606;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=369565;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#663\663]8;;\

           INFO     Detecting appropriate random factor                                             ]8;id=622198;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=893649;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1954\1954]8;;\

           INFO     Random factor is now appropriate                                                ]8;id=176937;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=510765;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#1972\1972]8;;\

           INFO     Meshing                                                                          ]8;id=899373;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=49794;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#675\675]8;;\

[10:18:25] INFO     Checking mesh conformity                                                        ]8;id=954008;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=893526;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#2003\2003]8;;\

...

[10:18:26] INFO     Checking connectivity                                                           ]8;id=640933;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=248726;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#2023\2023]8;;\

nodes:   0%|          | 0/1072 [00:00<?, ?it/s]

           INFO     Writing C:\Proyecto_QTCAD\meshes\sqdfdsoi.msh                                    ]8;id=115153;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=250536;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#730\730]8;;\

           INFO     Writing C:\Proyecto_QTCAD\meshes\sqdfdsoi.xao                                    ]8;id=841084;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py\builder.py]8;;\:]8;id=536618;file://c:\Users\aefra\anaconda3\envs\qtcad\Lib\site-packages\qtcad\builder\core\builder.py#730\730]8;;\



Waiting for Gmsh viewer to be closed...


## 4. Final considerations

In this tutorial, we walked through the complete process of creating the mesh and geometry files associated to an FD-SOI geometry created using QTCAD® Builder. Specifically, we:

1. Use a Builder-compatible `.oas` file to construct the geometry.
2. Generated the 3D geometry and heterostructure of a single quantum dot FD-SOI device.
3. Created a computational mesh and saved both the mesh (`.msh`) and geometry (`.xao`) files.

In the next tutorial, we will use the `.msh` and `.xao` files generated here to solve the Poisson and Schrödinger equation to simulate of the relevant electronic properties of the FD-SOI quantum device.